# Lesson 2: ReACT Framework with Real Tool Integration

## Understanding ReACT with Actual Tools

In this exercise, you'll implement **authentic ReACT (Reasoning and Acting)** where LLMs:
1. **Think** about what tools they need
2. **Act** by calling real investigation tools
3. **Observe** tool results and continue reasoning

**Key Difference**: Unlike basic ReACT that just describes actions, this version **actually executes tools** and processes real results.

**Time Required**: 15-20 minutes  
**Focus**: Practical ReACT implementation with financial investigation tools

## Setup: Import Pre-Built Investigation Tools

In [1]:
# Import standard libraries
from openai import OpenAI
import json

%load_ext autoreload
%autoreload 2

# TODO: Import our pre-built investigation tools
# Hint: Import get_transaction_history, get_customer_profile, check_regulatory_thresholds, process_tool_calls
# from investigation_tools import (...)

# TODO: Setup OpenAI client with Vocareum configuration
client = OpenAI(
     base_url="https://openai.vocareum.com/v1",
     api_key="voc-62787499117578256732026aedf382c0b8f6.70645548"  # This will be provided by Vocareum
 )

print("✅ Setup complete - ReACT tools ready!")
print("Available tools: transaction history, customer profiles, regulatory checks")

✅ Setup complete - ReACT tools ready!
Available tools: transaction history, customer profiles, regulatory checks


## Quick Tool Demo: See What We're Working With

In [2]:
import os
from dotenv import load_dotenv
from investigation_tools import get_transaction_history
from investigation_tools import get_customer_profile
from investigation_tools import check_regulatory_thresholds
from investigation_tools import process_tool_calls

# TODO: Quick demo of our investigation tools
# Test each tool to understand what data they return

print("🔍 TOOL DEMO - Transaction History:")
# TODO: Call get_transaction_history with account "high_risk_account_001" for 14 days
demo_transactions = get_transaction_history("high_risk_account_001",days=14)
print(json.dumps(demo_transactions, indent=2))

print("\n👤 TOOL DEMO - Customer Profile:")
# TODO: Call get_customer_profile for "CUST_001"
demo_customer = get_customer_profile("CUST_001")
print(json.dumps(demo_customer, indent=2))

print("\n📋 TOOL DEMO - Regulatory Check:")
# TODO: Call check_regulatory_thresholds for amount 9800, type "cash_deposit"
demo_check = check_regulatory_thresholds(9800,"cash_deposit")
print(json.dumps(demo_check, indent=2))

🔍 TOOL DEMO - Transaction History:
{
  "account_id": "high_risk_account_001",
  "period_days": 14,
  "transaction_count": 5,
  "transactions": [
    {
      "date": "2026-11-02",
      "type": "cash_deposit",
      "amount": 9800.0,
      "description": "Cash deposit - Branch 12"
    },
    {
      "date": "2024-01-04",
      "type": "cash_deposit",
      "amount": 9500.0,
      "description": "Cash deposit - Branch 7"
    },
    {
      "date": "2024-01-06",
      "type": "cash_deposit",
      "amount": 9900.0,
      "description": "Cash deposit - Branch 12"
    },
    {
      "date": "2024-01-09",
      "type": "cash_deposit",
      "amount": 9700.0,
      "description": "Cash deposit - Branch 3"
    },
    {
      "date": "2024-01-11",
      "type": "wire_out",
      "amount": 38000.0,
      "description": "Wire transfer - overseas account"
    }
  ]
}

👤 TOOL DEMO - Customer Profile:
{
  "customer_id": "CUST_001",
  "name": "John Smith",
  "account_id": "high_risk_account_001",
  "

## ReACT Prompt: Teaching LLM to Use Tools

In [3]:
import os
from dotenv import load_dotenv
from investigation_tools import get_transaction_history
from investigation_tools import get_customer_profile
from investigation_tools import check_regulatory_thresholds
from investigation_tools import process_tool_calls

load_dotenv('../../.env')
api_key = os.getenv("OPENAI_API_KEY")

# TODO: Create ReACT prompt that teaches the LLM to use our tools
REACT_PROMPT = """
You are a Financial Crimes Investigator using real investigation tools.

AVAILABLE TOOLS:
 1. get_transaction_history(account_id, days) - Get account transactions
 2. get_customer_profile(customer_id) - Get customer info and risk data  
 3. check_regulatory_thresholds(amount, type) - Check compliance requirements

PROCESS: Use THOUGHT → ACTION → OBSERVATION cycle:

THOUGHT: [Decide what you need to investigate]
ACTION: [Call tools using this exact JSON format]
```json
{
  "tool": "tool_name",
  "parameters": {"param1": "value1", "param2": "value2"}
}
```
OBSERVATION: [Analyze the tool results]

Continue until you have enough information for a final recommendation.
"""

print("📋 ReACT prompt configured for tool usage")

📋 ReACT prompt configured for tool usage


## Investigation Scenario: Suspicious Cash Deposits

In [4]:
# Investigation case for ReACT analysis
investigation_case = """
SUSPICIOUS ACTIVITY ALERT

Case: Multiple cash deposits just under $10,000
Customer ID: CUST_001
Account ID: high_risk_account_001
Time Period: Past 14 days
Alert Source: Branch manager noticed pattern
Customer Explanation: "Restaurant business doing really well"

TASK: Investigate and determine if SAR filing is required.
"""

print("📁 Investigation scenario loaded")
print("Focus: Potential cash structuring pattern")

📁 Investigation scenario loaded
Focus: Potential cash structuring pattern


## ReACT Investigation in Action

In [5]:
def run_react_investigation(case_details, max_rounds=3):
    """Run ReACT investigation with real tool integration"""
    
    print("🚀 STARTING ReACT INVESTIGATION")
    print("=" * 50)
    
    context = case_details
    
    for round_num in range(1, max_rounds + 1):
        print(f"\n🔄 ROUND {round_num}")
        print("-" * 30)
        
        # TODO: Create prompt combining REACT_PROMPT with case details
        prompt = f"""{REACT_PROMPT}

CASE DETAILS:
{context}

Conduct your investigation using THOUGHT → ACTION → OBSERVATION.
"""
        
        try:
             # TODO: Get LLM reasoning and tool calls
             response = client.chat.completions.create(
                 model= "gpt-4o",  # e.g., "gpt-4o"
                 messages=[
                    {"role":"system","content":REACT_PROMPT},
                    {"role":"user","content":prompt}
                 ],  # Construct messages with system and user roles
                 temperature=0.3,
                 max_tokens=600
             )
            
             llm_response = response.choices[0].message.content  # Extract LLM text response
             print("🤖 INVESTIGATOR RESPONSE:")
             print(llm_response)
             print()
            
            # TODO: Execute any tool calls using process_tool_calls
             tool_results = process_tool_calls(llm_response)  # Parse and execute tool calls from llm_response
            
            # TODO: Handle tool results and update context
             if tool_results:
                 # Add tool results to context for next round
                 context += f"\n\nROUND {round_num} TOOL RESULTS:\n"
                 for result in tool_results:
                     context += f"- {result['tool']}: {json.dumps(result['result'], indent=2)}\n"
             else:
                 # No tools called - likely final conclusion
                 print("✅ Investigation complete - no more tools needed")
                 break
                
                
        except Exception as e:
            print(f"❌ Error in Round {round_num}: {str(e)}")
            break
    
    print("\n" + "=" * 50)
    print("🏁 INVESTIGATION COMPLETE")

# TODO: Run the investigation once you've implemented the function
run_react_investigation(investigation_case)
print("ReACT investigation function ready to implement")

🚀 STARTING ReACT INVESTIGATION

🔄 ROUND 1
------------------------------
🤖 INVESTIGATOR RESPONSE:
THOUGHT: The case involves multiple cash deposits just under $10,000, which could be indicative of structuring to avoid regulatory reporting requirements. First, I need to review the transaction history for the account over the past 14 days to understand the pattern and frequency of these deposits.

ACTION: 
```json
{
  "tool": "get_transaction_history",
  "parameters": {"account_id": "high_risk_account_001", "days": 14}
}
```

🔧 Executing get_transaction_history({'account_id': 'high_risk_account_001', 'days': 14})
📊 Result: {
  "account_id": "high_risk_account_001",
  "period_days": 14,
  "transaction_count": 5,
  "transactions": [
    {
      "date": "2026-11-02",
      "type": "cash_deposit",
      "amount": 9800.0,
      "description": "Cash deposit - Branch 12"
    },
    {
      "date": "2024-01-04",
      "type": "cash_deposit",
      "amount": 9500.0,
      "description": "Cash dep

## Compare: Basic vs ReACT with Tools

Let's see the difference between basic analysis and ReACT with real tools:

In [12]:
# TODO: Compare basic analysis without tools vs ReACT with tools
print("📊 COMPARISON: Basic Analysis vs ReACT with Tools\n")

print("🔸 BASIC ANALYSIS (No Tools):")
basic_prompt = "analyze the case and explain any suspicious activity without using tools or external data" #TODO: Create a simple prompt for basic analysis

# TODO: Get basic response from OpenAI without tools
basic_response = client.chat.completions.create(
                  model= "gpt-4o",  # e.g., "gpt-4o"
                  messages=[
                    {"role":"system","content":basic_prompt},
                    {"role":"user","content":investigation_case}
                 ],
                 temperature=0.3,
                 max_tokens=600
 )

print(basic_response.choices[0].message.content)

print("\n" + "="*60)
print("🔸 ReACT WITH TOOLS (Above):")
print("- Used real transaction data")
print("- Checked actual customer profile")
print("- Verified regulatory thresholds")
print("- Made data-driven decisions")
print("\n✅ ReACT with tools provides evidence-based investigation!")

📊 COMPARISON: Basic Analysis vs ReACT with Tools

🔸 BASIC ANALYSIS (No Tools):
In this case, the suspicious activity alert is triggered by multiple cash deposits just under $10,000 within a 14-day period. This pattern is often associated with a practice known as "structuring" or "smurfing," where an individual deliberately makes multiple smaller deposits to avoid triggering the bank's obligation to report transactions over $10,000 to authorities as required by the Bank Secrecy Act (BSA).

Here are the key points to consider in this investigation:

1. **Pattern of Deposits**: The fact that the deposits are consistently just under $10,000 is a red flag. This suggests a potential attempt to evade the reporting threshold intentionally.

2. **Frequency and Amount**: Multiple deposits in a short time frame, especially if they are close to the $10,000 mark, further suggest structuring. It would be important to review the exact amounts and frequency to see if there is a clear pattern.

3. **Cu

## Test Your Own Scenario

In [7]:
# Template for testing different scenarios
custom_scenario = """
INVESTIGATION CASE: [Your Case Title]

Case: [Description of suspicious activity]
Customer ID: [Choose CUST_001, CUST_002, or CUST_003]
Account ID: [Choose high_risk_account_001, business_account_002, or normal_account]
Alert Source: [How was this detected]
Customer Explanation: [What customer says]

TASK: [What you want the investigator to determine]
"""

print("💡 TESTING FRAMEWORK:")
print("1. Fill in the template above")
print("2. Run: run_react_investigation(your_scenario)")
print("3. Watch ReACT use tools to investigate")
print("\nAvailable test combinations:")
print("- CUST_001 + high_risk_account_001: Restaurant manager, cash deposits")
print("- CUST_002 + business_account_002: Business owner, wire transfers")
print("- CUST_003 + normal_account: Software engineer, normal transactions")

# TODO: Try creating your own scenario and testing it!

💡 TESTING FRAMEWORK:
1. Fill in the template above
2. Run: run_react_investigation(your_scenario)
3. Watch ReACT use tools to investigate

Available test combinations:
- CUST_001 + high_risk_account_001: Restaurant manager, cash deposits
- CUST_002 + business_account_002: Business owner, wire transfers
- CUST_003 + normal_account: Software engineer, normal transactions


## Key Takeaways: Real ReACT Implementation

### What You'll Demonstrate:

**1. Authentic Tool Integration**
- ✅ Real function calls with actual parameters
- ✅ JSON parsing and tool execution
- ✅ Tool results fed back into reasoning

**2. Practical ReACT Workflow**
- ✅ LLM decides what tools to use based on investigation needs
- ✅ Tools provide real data for analysis
- ✅ Iterative refinement based on tool results

**3. Financial Investigation Benefits**
- ✅ Evidence-based decisions vs. speculation
- ✅ Systematic data gathering
- ✅ Regulatory compliance checking
- ✅ Audit trail of investigation steps

### vs. Basic ReACT (Simulated Actions):
- **Basic ReACT**: "I would check the transaction history" ❌
- **ReACT with Tools**: Actually retrieves and analyzes transaction data ✅

### Real-World Applications:
- **Financial Compliance**: Automated suspicious activity investigation
- **Risk Assessment**: Data-driven customer risk evaluation
- **Regulatory Reporting**: Evidence gathering for SAR/CTR filings
- **Case Management**: Structured investigation workflows

**The Power**: ReACT with real tools transforms LLMs from "thinking about actions" to "actually taking actions" with real data and measurable outcomes! 🚀

---
*Time: ~15-20 minutes | Focus: Practical ReACT Implementation | Context: Financial Investigation*